# Speculative Decoding Benchmark

This notebook demonstrates speculative decoding, a technique that can provide 1.5-2.5x speedup in inference with **no quality degradation**.

## How It Works

1. A small "draft" model generates K tokens quickly
2. The main "target" model verifies all K tokens in parallel
3. Accepted tokens are kept, rejected tokens are regenerated
4. Net result: faster inference without quality loss

## Setup

In [ ]:
import sys
sys.path.append('..')

from benchmark_speculative_decoding import run_speculative_decoding_benchmark
from visualize_results import create_all_visualizations
import json
import pandas as pd

## Run Benchmarks

We'll compare:
- **Standard inference**: Target model alone
- **Speculative decoding**: Target + draft model

In [ ]:
# Configuration
TARGET_MODEL = "facebook/opt-1.3b"  # Main model
DRAFT_MODEL = "facebook/opt-350m"   # Smaller, faster model
PROMPT = "The future of artificial intelligence is"
MAX_TOKENS = 100
NUM_RUNS = 5
LOOKAHEAD = 4  # Number of tokens draft model generates
OUTPUT_FILE = "speculative_decoding_results.json"

In [ ]:
# Run the benchmark
print("Starting speculative decoding benchmarks...")
print(f"Target Model: {TARGET_MODEL}")
print(f"Draft Model: {DRAFT_MODEL}")
print(f"This may take several minutes...\n")

results = run_speculative_decoding_benchmark(
    target_model_name=TARGET_MODEL,
    draft_model_name=DRAFT_MODEL,
    prompt=PROMPT,
    max_new_tokens=MAX_TOKENS,
    num_runs=NUM_RUNS,
    lookahead=LOOKAHEAD,
    output_file=OUTPUT_FILE
)

## Analyze Results

In [ ]:
# Load results as DataFrame
with open(OUTPUT_FILE, 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)
df

In [ ]:
# Compare standard vs speculative
print("\n" + "="*60)
print("SPECULATIVE DECODING COMPARISON")
print("="*60)

if len(df) >= 2:
    std = df[df['speculative_decoding'] == False].iloc[0]
    spec = df[df['speculative_decoding'] == True].iloc[0]
    
    print(f"\nSTANDARD INFERENCE:")
    print(f"  Time: {std['inference_time']:.2f}s")
    print(f"  Tokens/sec: {std['tokens_per_second']:.1f}")
    
    print(f"\nSPECULATIVE DECODING:")
    print(f"  Time: {spec['inference_time']:.2f}s")
    print(f"  Tokens/sec: {spec['tokens_per_second']:.1f}")
    print(f"  Speedup: {spec['speedup']:.2f}x")
    
    improvement = ((std['inference_time'] - spec['inference_time']) / std['inference_time']) * 100
    print(f"\n🚀 {improvement:.1f}% faster with speculative decoding!")

## Generate Visualizations

In [ ]:
# Create visualizations
create_all_visualizations(
    results_file=OUTPUT_FILE,
    output_dir="visualizations_spec",
    gpu_cost_per_hour=1.10
)

In [ ]:
# Display visualizations
from IPython.display import Image, display

print("\n⚡ Speculative Decoding Performance:")
display(Image('visualizations_spec/speculative_decoding.png'))

## Cost Savings with Speculative Decoding

In [ ]:
# Cost analysis
if len(df) >= 2:
    GPU_COST_PER_HOUR = 1.10
    INFERENCES_PER_MONTH = 100_000_000
    
    std = df[df['speculative_decoding'] == False].iloc[0]
    spec = df[df['speculative_decoding'] == True].iloc[0]
    
    std_cost = (std['inference_time'] / 3600) * GPU_COST_PER_HOUR * INFERENCES_PER_MONTH
    spec_cost = (spec['inference_time'] / 3600) * GPU_COST_PER_HOUR * INFERENCES_PER_MONTH
    
    savings = std_cost - spec_cost
    savings_pct = (savings / std_cost) * 100
    
    print("\n" + "="*60)
    print(f"MONTHLY COST PROJECTION ({INFERENCES_PER_MONTH:,} inferences)")
    print("="*60)
    print(f"\nStandard Inference: ${std_cost:,.2f}")
    print(f"Speculative Decoding: ${spec_cost:,.2f}")
    print(f"\n💰 Monthly Savings: ${savings:,.2f} ({savings_pct:.1f}%)")

## Understanding the Trade-offs

### Advantages ✅
- **1.5-2.5x speedup**: Significantly faster inference
- **No quality loss**: Same model, same outputs
- **Easy to implement**: Works with existing models

### Considerations ⚠️
- **Memory overhead**: Need to load both draft and target models (+10-20% VRAM)
- **Draft model selection**: Best with models from same family
- **Acceptance rate**: Speedup depends on draft model quality

## Tuning Parameters

Key parameters to optimize:

In [ ]:
# Experiment with different lookahead values
lookahead_values = [2, 4, 8]

print("Recommended lookahead settings:")
print("  - Small models (< 1B): lookahead = 2-4")
print("  - Medium models (1-7B): lookahead = 4-6")
print("  - Large models (> 7B): lookahead = 6-8")
print("\nHigher lookahead = more tokens verified in parallel")
print("But too high = lower acceptance rate")

## Combining with Quantization

You can combine speculative decoding with quantization for even better results:

In [ ]:
print("Combined Optimizations:")
print("\n1. 4-bit Target + FP16 Draft:")
print("   - ~60% VRAM savings from quantization")
print("   - ~1.5-2x speedup from spec-decode")
print("   - Total: 3-4x better efficiency")
print("\n2. 8-bit Target + 4-bit Draft:")
print("   - ~50% VRAM savings")
print("   - ~1.5-2x speedup")
print("   - Better quality preservation")

## Key Takeaways

### When to Use Speculative Decoding
- ✅ Latency-critical applications
- ✅ High-throughput inference servers
- ✅ When you have extra VRAM for draft model
- ✅ Cost reduction without quality loss

### Best Practices
- Use draft model that's 3-4x smaller than target
- Same model family (e.g., OPT-350M drafts for OPT-1.3B)
- Tune lookahead based on acceptance rate
- Combine with quantization for maximum efficiency

## Next Steps

1. Try with different model combinations
2. Measure acceptance rates for your use case
3. Tune lookahead parameter
4. Combine with quantization (see quantization_demo.ipynb)
5. Deploy to production inference server